# 02 — Create 20 paired train/test batches and method-specific transformed matrices

Subjects are split into 20 repeated train/test batches so all samples from one subject remain together. The same saved batches are consumed by both TEMPTED and MEFISTO.

Feature filtering uses **training data only**:

- prevalence: present (`count > 0`) in at least 10% of training samples
- abundance: has at least 1% relative abundance in at least 10% of training samples

**TEMPTED input:** after train-only feature filtering, for each sample, set the pseudocount to **one-half of that sample's smallest nonzero taxon value**, then apply CLR.

**MEFISTO input:** apply rCLR without a pseudocount. Original zeros remain missing (`NaN`) and are passed to MEFISTO as missing values.


In [1]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

NUMBER_OF_BATCHES = 20
TRAIN_FRACTION = 0.70
MINIMUM_PREVALENCE = 0.1
MINIMUM_RELATIVE_ABUNDANCE = 0.01
RANDOM_SEED = 2026

root = Path(".") if Path("data").exists() else Path("..")
dataset = sorted(
    path for path in (root / "data" / "processed_16s").iterdir()
    if path.is_dir()
    and (path / "counts.csv").exists()
    and (path / "metadata.csv").exists()
)[-1]

output = root / "data" / "splits" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

counts = pd.read_csv(dataset / "counts.csv", dtype={"sample_id": str}).set_index("sample_id")
metadata = pd.read_csv(
    dataset / "metadata.csv",
    dtype={"sample_id": str, "subject_id": str, "label": str},
)
counts = counts.loc[metadata["sample_id"]]
subjects = metadata.drop_duplicates("subject_id")[["subject_id", "label"]]

print("Input:", dataset)
print("Output:", output)


Input: ../data/processed_16s/20260809_183824
Output: ../data/splits/20260809_183835


## Method-specific preprocessing

### TEMPTED: sample-specific half-minimum pseudocount then CLR

For each sample, the pseudocount is one-half of that sample's smallest nonzero retained taxon value. This matches the TEMPTED package's documented default rule. CLR is then calculated from that sample.

### MEFISTO: rCLR with missing zeros

MEFISTO does not use the TEMPTED pseudocount. rCLR is computed from positive entries within each sample, while original zeros remain missing.


In [2]:
def relative_abundance(x):
    return x.div(x.sum(axis=1), axis=0)


def clr_half_minimum(x):
    """TEMPTED-style CLR: each sample gets half its own smallest positive value."""
    values = x.to_numpy(float)
    out = np.empty_like(values, dtype=float)
    pseudocounts = np.empty(len(values), dtype=float)

    for i, row in enumerate(values):
        positive = row[row > 0]
        if len(positive) == 0:
            raise ValueError(f"Sample {x.index[i]} has no positive taxa after filtering.")
        pseudocount = positive.min() / 2
        pseudocounts[i] = pseudocount
        logged = np.log(row + pseudocount)
        out[i] = logged - logged.mean()

    return pd.DataFrame(out, index=x.index, columns=x.columns), pseudocounts


def rclr(x):
    values = x.to_numpy(float)
    transformed = np.full(values.shape, np.nan)

    for row_number, row in enumerate(values):
        positive = row > 0
        logged = np.log(row[positive])
        transformed[row_number, positive] = logged - logged.mean()

    return pd.DataFrame(transformed, index=x.index, columns=x.columns)


In [3]:
rows = []

for number in range(1, NUMBER_OF_BATCHES + 1):
    train_subjects, test_subjects = train_test_split(
        subjects,
        train_size=TRAIN_FRACTION,
        stratify=subjects["label"],
        random_state=RANDOM_SEED + number,
    )

    train_meta = metadata[metadata["subject_id"].isin(train_subjects["subject_id"])].copy()
    test_meta = metadata[metadata["subject_id"].isin(test_subjects["subject_id"])].copy()
    train_counts = counts.loc[train_meta["sample_id"]]
    test_counts = counts.loc[test_meta["sample_id"]]

    # Benchmark filter: genus must be >=1% relative abundance in >=10% of
    # TRAINING samples. Test samples never determine which genera are retained.
    relative = relative_abundance(train_counts)
    keep = (relative >= MINIMUM_RELATIVE_ABUNDANCE).mean(axis=0) >= MINIMUM_PREVALENCE

    features = train_counts.columns[keep]
    train_counts = train_counts[features]
    test_counts = test_counts[features]

    train_tempted_clr, train_pseudocounts = clr_half_minimum(train_counts)
    test_tempted_clr, test_pseudocounts = clr_half_minimum(test_counts)
    train_mefisto_rclr = rclr(train_counts)
    test_mefisto_rclr = rclr(test_counts)

    folder = output / f"batch_{number:03d}"
    folder.mkdir()

    for name, table in {
        "train_counts": train_counts,
        "test_counts": test_counts,
        "train_tempted_clr": train_tempted_clr,
        "test_tempted_clr": test_tempted_clr,
        "train_mefisto_rclr": train_mefisto_rclr,
        "test_mefisto_rclr": test_mefisto_rclr,
    }.items():
        table.rename_axis("sample_id").reset_index().to_csv(
            folder / f"{name}.csv.gz",
            index=False,
        )

    train_meta.to_csv(folder / "train_metadata.csv.gz", index=False)
    test_meta.to_csv(folder / "test_metadata.csv.gz", index=False)

    rows.append([
        output.name,
        folder.name,
        train_meta["subject_id"].nunique(),
        test_meta["subject_id"].nunique(),
        len(features),
        float(np.median(train_pseudocounts)),
        int(train_mefisto_rclr.isna().sum().sum()),
        int(test_mefisto_rclr.isna().sum().sum()),
    ])

summary = pd.DataFrame(
    rows,
    columns=[
        "split_run",
        "batch",
        "train_subjects",
        "test_subjects",
        "features",
        "median_train_tempted_pseudocount",
        "missing_train_mefisto_rclr",
        "missing_test_mefisto_rclr",
    ],
)
summary.to_csv(output / "batch_summary.csv", index=False)

# Explicit split manifest for downstream pairing.
manifest = []
for folder in sorted(path for path in output.glob("batch_*") if path.is_dir()):
    train_meta = pd.read_csv(folder / "train_metadata.csv.gz", dtype={"subject_id": str})
    test_meta = pd.read_csv(folder / "test_metadata.csv.gz", dtype={"subject_id": str})
    for role, ids in [("train", train_meta["subject_id"]), ("test", test_meta["subject_id"])]:
        for subject_id in sorted(ids.unique()):
            manifest.append({
                "split_run": output.name,
                "batch": folder.name,
                "set": role,
                "subject_id": subject_id,
            })

pd.DataFrame(manifest).to_csv(output / "subject_split_manifest.csv", index=False)

print("Saved:", output)
summary


Saved: ../data/splits/20260809_183835


,split_run,batch,train_subjects,test_subjects,features,median_train_tempted_pseudocount,missing_train_mefisto_rclr,missing_test_mefisto_rclr
0,20260809_183835,batch_001,146,63,31,0.000017,5180,1989
1,20260809_183835,batch_002,146,63,31,0.000017,4954,2215
2,20260809_183835,batch_003,146,63,32,0.000017,5849,2296
3,20260809_183835,batch_004,146,63,31,0.000017,5093,2076
4,20260809_183835,batch_005,146,63,31,0.000018,5118,2051
5,20260809_183835,batch_006,146,63,32,0.000017,5243,2171
6,20260809_183835,batch_007,146,63,31,0.000017,5015,2154
7,20260809_183835,batch_008,146,63,32,0.000016,5618,2033
8,20260809_183835,batch_009,146,63,32,0.000017,5417,2179
9,20260809_183835,batch_010,146,63,31,0.000017,4944,2225
